# 安全帽检测 - 两阶段方案（使用训练模型）

方案：
1. **第一阶段**：使用 DEIMv2 COCO 预训练模型检测人
2. **第二阶段**：使用训练好的 best_stg1.pth 模型检测安全帽

In [ ]:
import torch
import cv2
import numpy as np
import os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision.transforms as T
import urllib.request

import sys
DEIMV2_PATH = r'D:\\AI\\Git\\DEIMv2'
sys.path.append(DEIMV2_PATH)

from engine.core import YAMLConfig
from engine.deim import PostProcessor

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

In [ ]:
# 第一阶段：COCO人员检测模型
COCO_CONFIG_PATH = r'D:\\AI\\Git\\DEIMv2\\configs\\deimv2\\deimv2_dinov3_s_coco.yml'

# 第二阶段：安全帽检测模型
HELMET_CONFIG_PATH = r'D:\\AI\\Git\\DEIMv2\\configs\\deimv2\\deimv2_hgnetv2_atto_helmet_cpu3.yml'
HELMET_MODEL_PATH = r'D:\\AI\\Git\\DEIMv2\\outputs\\deimv2_hgnetv2_atto_helmet_gpu_3060\\best_stg1.pth'

INPUT_DIR = r'D:\\AI\\Datasets\\sy-person'
OUTPUT_DIR = r'./output_two_stage_helmet'

PERSON_THRESHOLD = 0.4
HELMET_THRESHOLD = 0.05
PERSON_CLASS_ID = 0

HELMET_CLASSES = {0: 'Helmet', 1: 'NoHelmet', 2: 'NoVest', 3: 'Vest'}

COLORS = {'person': '#00BFFF', 'helmet': '#00FF00', 'no_helmet': '#FF0000', 'unknown': '#808080'}

print(f'COCO配置: {COCO_CONFIG_PATH}')
print(f'安全帽模型: {HELMET_MODEL_PATH}')

In [ ]:
# ========== 第一阶段：加载COCO人员检测模型 ==========
print('正在加载COCO人员检测模型...')

cfg_coco = YAMLConfig(COCO_CONFIG_PATH)
eval_spatial_size_coco = cfg_coco.yaml_cfg.get('eval_spatial_size', [640, 640])
print(f'COCO评估尺寸: {eval_spatial_size_coco}')

person_model = cfg_coco.model

# 下载/加载COCO预训练权重
cache_dir = Path.home() / '.cache' / 'deimv2'
cache_dir.mkdir(parents=True, exist_ok=True)
coco_weights_path = cache_dir / 'deimv2_dinov3_s_coco.pth'

if not coco_weights_path.exists():
    print('从HuggingFace下载预训练权重...')
    url = 'https://huggingface.co/Intellindust/DEIMv2_DINOv3_S_COCO/resolve/main/model.pth'
    try:
        urllib.request.urlretrieve(url, str(coco_weights_path))
        print(f'下载完成: {coco_weights_path}')
    except Exception as e:
        print(f'下载失败: {e}')

if coco_weights_path.exists():
    checkpoint = torch.load(str(coco_weights_path), map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    new_state_dict = {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}
    person_model.load_state_dict(new_state_dict, strict=False)
    print('✅ COCO预训练权重加载成功！')
else:
    print('⚠️ 未找到预训练权重！')

person_model = person_model.to(device)
person_model.eval()
person_postprocessor = cfg_coco.postprocessor
person_postprocessor.eval_spatial_size = eval_spatial_size_coco

# ========== 第二阶段：加载安全帽检测模型 ==========
print('\n加载安全帽检测模型...')
cfg_helmet = YAMLConfig(HELMET_CONFIG_PATH)
eval_spatial_size_helmet = cfg_helmet.yaml_cfg.get('eval_spatial_size', [640, 640])
helmet_model = cfg_helmet.model

checkpoint = torch.load(HELMET_MODEL_PATH, map_location='cpu', weights_only=False)
state_dict = checkpoint.get('model', checkpoint.get('ema', {}).get('module', checkpoint))
new_state_dict = {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}
helmet_model.load_state_dict(new_state_dict, strict=False)
helmet_model = helmet_model.to(device)
helmet_model.eval()

helmet_postprocessor = cfg_helmet.postprocessor
helmet_postprocessor.eval_spatial_size = eval_spatial_size_helmet
print('✅ 安全帽检测模型加载成功！')

In [ ]:
class ImagePreprocessor:
    def __init__(self, target_size=(640, 640)):
        self.target_size = target_size
    def __call__(self, image):
        if isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        orig_w, orig_h = image.size
        image = image.resize(self.target_size)
        image_tensor = T.ToTensor()(image)
        normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        image_tensor = normalize(image_tensor)
        return image_tensor, (orig_w, orig_h)

preprocessor_coco = ImagePreprocessor(target_size=tuple(eval_spatial_size_coco))
preprocessor_helmet = ImagePreprocessor(target_size=tuple(eval_spatial_size_helmet))
print('预处理器初始化完成')

In [ ]:
def detect_persons(image_path, model, postprocessor, preprocessor, device, threshold=0.4):
    orig_image = Image.open(image_path).convert('RGB')
    image_tensor, (orig_w, orig_h) = preprocessor(orig_image)
    image_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(image_tensor)
    orig_target_sizes = torch.tensor([[orig_h, orig_w]]).to(device)
    results = postprocessor(outputs, orig_target_sizes)[0]
    persons = []
    boxes = results['boxes'].cpu().numpy()
    scores = results['scores'].cpu().numpy()
    labels = results['labels'].cpu().numpy()
    for box, score, label in zip(boxes, scores, labels):
        if score >= threshold and int(label) == PERSON_CLASS_ID:
            persons.append({'bbox': [int(x) for x in box], 'score': float(score)})
    return persons, orig_image

def detect_helmets_in_crop(crop, model, postprocessor, preprocessor, device, threshold=0.05):
    image_tensor, (orig_w, orig_h) = preprocessor(crop)
    image_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(image_tensor)
    orig_target_sizes = torch.tensor([[orig_h, orig_w]]).to(device)
    results = postprocessor(outputs, orig_target_sizes)[0]
    helmets = []
    for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
        if score >= threshold and int(label) in [0, 1]:
            helmets.append({'class_id': int(label), 'score': float(score)})
    return helmets

def process_image(image_path, output_path=None):
    persons, orig_image = detect_persons(image_path, person_model, person_postprocessor, preprocessor_coco, device, PERSON_THRESHOLD)
    result = orig_image.copy()
    draw = ImageDraw.Draw(result)
    try:
        font = ImageFont.truetype('C:/Windows/Fonts/simhei.ttf', 20)
    except:
        font = ImageFont.load_default()
    stats = {'persons': len(persons), 'with_helmet': 0, 'without_helmet': 0}
    for person in persons:
        x1, y1, x2, y2 = person['bbox']
        w, h = x2 - x1, y2 - y1
        cx1, cy1 = max(0, x1 - int(w*0.1)), max(0, y1 - int(h*0.2))
        cx2, cy2 = min(orig_image.width, x2 + int(w*0.1)), min(orig_image.height, y2 + int(h*0.1))
        if cx2 <= cx1 or cy2 <= cy1:
            continue
        crop = orig_image.crop((cx1, cy1, cx2, cy2))
        helmets = detect_helmets_in_crop(crop, helmet_model, helmet_postprocessor, preprocessor_helmet, device, HELMET_THRESHOLD)
        status = 'unknown'
        best_conf = 0
        for h in helmets:
            if h['score'] > best_conf:
                best_conf = h['score']
                status = 'helmet' if h['class_id'] == 0 else 'no_helmet'
        if status == 'helmet':
            stats['with_helmet'] += 1
        elif status == 'no_helmet':
            stats['without_helmet'] += 1
        color = COLORS.get(status, COLORS['unknown'])
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        label = f"Person {person['score']:.2f}"
        if status != 'unknown':
            label += f" | {HELMET_CLASSES[0 if status=='helmet' else 1]}"
        text_bbox = draw.textbbox((x1, y1-25), label, font=font)
        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, y1-25), label, fill='white', font=font)
    if output_path:
        result.save(output_path)
    return result, stats

print('检测函数定义完成')

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
image_files = list(Path(INPUT_DIR).glob('*.png')) + list(Path(INPUT_DIR).glob('*.jpg'))

if len(image_files) == 0:
    print('没有找到测试图像')
else:
    test_image = str(image_files[0])
    print(f'测试: {test_image}')
    result, stats = process_image(test_image, os.path.join(OUTPUT_DIR, f'result_{Path(test_image).name}'))
    print(f'人员: {stats["persons"]}, 有安全帽: {stats["with_helmet"]}, 无: {stats["without_helmet"]}')
    plt.figure(figsize=(12, 8))
    plt.imshow(result)
    plt.axis('off')
    plt.show()

In [ ]:
total = {'images': 0, 'persons': 0, 'with_helmet': 0, 'without_helmet': 0}
for img_path in tqdm(image_files[:10], desc='处理'):
    try:
        _, stats = process_image(str(img_path), os.path.join(OUTPUT_DIR, f'result_{img_path.name}'))
        total['images'] += 1
        total['persons'] += stats['persons']
        total['with_helmet'] += stats['with_helmet']
        total['without_helmet'] += stats['without_helmet']
    except Exception as e:
        print(f'错误 {img_path.name}: {e}')
print(f'\n处理{total["images"]}张图, {total["persons"]}人, 有安全帽{total["with_helmet"]}, 无{total["without_helmet"]}')